In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("data/raw/gatrout_history.csv")

print("Rows:" , len(df))
print ("columns", df.columns.tolist())


Rows: 500
columns ['id', 'stocked_on', 'waterbody_id', 'county', 'waterbody', 'species', 'quantity']


In [4]:
print("Rows:" , len(df))
print("unique streams:", df["waterbody"].nunique())
print("earliest:", df["stocked_on"].min())
print("latest:", df["stocked_on"].max())

Rows: 500
unique streams: 86
earliest: 2026-05-12
latest: 2026-08-14


In [5]:
stream_counts = (
    df.groupby(["waterbody", "county"])
    .size()
    .reset_index(name ="stocking_count" )
    .sort_values("stocking_count", ascending=False)

)

stream_counts.head(20)

,waterbody,county,stocking_count
41,Moccasin Creek,Rabun,24
52,Rock Creek (F),Fannin,15
17,Cooper Creek,Union,14
38,Low Gap Creek,White,14
4,Blueridge TW,Fannin,14
19,Dicks Creek,Lumpkin,14
57,Smith Creek,White,14
68,Tallulah River (T),Towns,13
67,Tallulah River (R),Rabun,13
32,Jasus Creek,White,13


In [6]:
df ["stocked_on"] =pd.to_datetime(df["stocked_on"])
df = df.sort_values(["waterbody", "county", "stocked_on"])
df["days_since_previous"] =(
    df.groupby(["waterbody", "county"])["stocked_on"]
    .diff()
    .dt.days
)

df[["waterbody", "county", "stocked_on", "days_since_previous"]].head(20)

,waterbody,county,stocked_on,days_since_previous
404,Allison Creek,Dade,2026-05-26,NaN
238,Allison Creek,Dade,2026-06-24,29.0
422,Amicalola Creek,Dawson,2026-05-22,NaN
339,Amicalola Creek,Dawson,2026-06-04,13.0
231,Amicalola Creek,Dawson,2026-06-25,21.0
124,Amicalola Creek,Dawson,2026-07-16,21.0
377,Big Creek,Fannin,2026-05-29,NaN
228,Big Creek,Fannin,2026-06-25,27.0
92,Big Creek,Fannin,2026-07-23,28.0
426,Black Rock Lake,Rabun,2026-05-21,NaN


In [7]:
interval_stats = (
    df.groupby(["waterbody", "county"])["days_since_previous"]
      .agg(["count", "mean" , "median", "min" , "max"])
      .reset_index()
)
interval_stats.sort_values("count", ascending = False).head(20)

,waterbody,county,count,mean,median,min,max
41,Moccasin Creek,Rabun,23,3.956522,4.0,3.0,8.0
52,Rock Creek (F),Fannin,14,6.500000,6.0,1.0,11.0
17,Cooper Creek,Union,13,6.461538,7.0,2.0,11.0
38,Low Gap Creek,White,13,7.230769,7.0,5.0,9.0
4,Blueridge TW,Fannin,13,6.230769,6.0,2.0,8.0
19,Dicks Creek,Lumpkin,13,7.000000,7.0,6.0,8.0
57,Smith Creek,White,13,7.230769,7.0,5.0,10.0
68,Tallulah River (T),Towns,12,7.666667,7.5,5.0,13.0
67,Tallulah River (R),Rabun,12,7.666667,7.5,5.0,13.0
32,Jasus Creek,White,12,7.166667,7.0,5.0,9.0


In [8]:
latest_stockings =(
    df.groupby(["waterbody", "county"])["stocked_on"]
      .max()
      .reset_index()
      .rename(columns={"stocked_on":"last_stocked"})

)
latest_stockings.head(20)

,waterbody,county,last_stocked
0,Allison Creek,Dade,2026-06-24
1,Amicalola Creek,Dawson,2026-07-16
2,Big Creek,Fannin,2026-07-23
3,Black Rock Lake,Rabun,2026-05-21
4,Blueridge TW,Fannin,2026-08-11
5,Boggs Creek,Lumpkin,2026-08-13
6,Brasstown Creek (T),Towns,2026-07-01
7,Brasstown Creek (U),Union,2026-07-01
8,Canada Creek,Union,2026-05-19
9,Cartecay River,Gilmer,2026-06-02


In [9]:
stream_summary = latest_stockings.merge(
    interval_stats,
    on=["waterbody","county" ],
    how ="left"

)
stream_summary.head(20)

,waterbody,county,last_stocked,count,mean,median,min,max
0,Allison Creek,Dade,2026-06-24,1,29.000000,29.0,29.0,29.0
1,Amicalola Creek,Dawson,2026-07-16,3,18.333333,21.0,13.0,21.0
2,Big Creek,Fannin,2026-07-23,2,27.500000,27.5,27.0,28.0
3,Black Rock Lake,Rabun,2026-05-21,0,NaN,NaN,NaN,NaN
4,Blueridge TW,Fannin,2026-08-11,13,6.230769,6.0,2.0,8.0
5,Boggs Creek,Lumpkin,2026-08-13,12,7.583333,7.0,6.0,13.0
6,Brasstown Creek (T),Towns,2026-07-01,1,44.000000,44.0,44.0,44.0
7,Brasstown Creek (U),Union,2026-07-01,1,44.000000,44.0,44.0,44.0
8,Canada Creek,Union,2026-05-19,0,NaN,NaN,NaN,NaN
9,Cartecay River,Gilmer,2026-06-02,0,NaN,NaN,NaN,NaN


In [15]:
today = pd.Timestamp("2026-08-19")

stream_summary["days_since_last"] = (
    today - stream_summary["last_stocked"]
).dt.days

In [17]:
print(stream_summary.columns)

Index(['waterbody', 'county', 'last_stocked', 'count', 'mean', 'median', 'min',
       'max', 'date_since_last', 'days_since_last'],
      dtype='str')


In [18]:
stream_summary.sort_values(
    "days_since_last",
    ascending=False
).head(20)

,waterbody,county,last_stocked,count,mean,median,min,max,date_since_last,days_since_last
71,Timpson,Rabun,2026-05-15,0,NaN,NaN,NaN,NaN,96,96
14,Clear Creek,Gilmer,2026-05-19,0,NaN,NaN,NaN,NaN,92,92
18,Coosawattee River,Gilmer,2026-05-19,0,NaN,NaN,NaN,NaN,92,92
77,Turkey Creek,Gilmer,2026-05-19,0,NaN,NaN,NaN,NaN,92,92
23,Ellijay River,Gilmer,2026-05-19,0,NaN,NaN,NaN,NaN,92,92
8,Canada Creek,Union,2026-05-19,0,NaN,NaN,NaN,NaN,92,92
63,Suches Creek,Union,2026-05-20,0,NaN,NaN,NaN,NaN,91,91
12,Chestatee River,Lumpkin,2026-05-21,0,NaN,NaN,NaN,NaN,90,90
3,Black Rock Lake,Rabun,2026-05-21,0,NaN,NaN,NaN,NaN,90,90
61,Stonewall Creek,Rabun,2026-05-21,0,NaN,NaN,NaN,NaN,90,90


In [21]:
stream_summary["due_ratio"] = (
    stream_summary["days_since_last"] / stream_summary["median"]
)

In [22]:
stream_summary.sort_values(
    "due_ratio",
    ascending=False
).head(20)

,waterbody,county,last_stocked,count,mean,median,min,max,date_since_last,days_since_last,due_ratio
49,Panther Creek (S),Stephens,2026-06-02,3,6.333333,6.0,4.0,9.0,78,78,13.000000
39,Middle Broad River,Stephens,2026-06-11,3,9.333333,9.0,6.0,13.0,69,69,7.666667
85,Wolf Creek,Union,2026-07-01,5,9.800000,9.0,5.0,16.0,49,49,5.444444
48,Panther Creek (H),Habersham,2026-06-29,4,9.750000,10.0,5.0,14.0,51,51,5.100000
84,Winfield Scott Lake,Union,2026-06-15,2,13.000000,13.0,12.0,14.0,65,65,5.000000
40,Mill Creek - 1,Murray,2026-06-18,2,15.000000,15.0,14.0,16.0,62,62,4.133333
16,Connesena Creek,Bartow,2026-06-30,4,11.750000,13.0,6.0,15.0,50,50,3.846154
62,Storey Mill Creek,Chattooga,2026-06-15,1,17.000000,17.0,17.0,17.0,65,65,3.823529
54,Ruff Creek,Chattooga,2026-06-15,1,17.000000,17.0,17.0,17.0,65,65,3.823529
28,Hiawassee River,Towns,2026-07-01,3,14.666667,14.0,14.0,16.0,49,49,3.500000
